# Notebook 4 — Machine Learning Clássico (Modelo Personalizado)

## Protocolo de avaliação

Cada modelo é **personalizado por paciente**: treinado e testado nos dados do
mesmo paciente, nunca misturando pacientes entre si.

**Split pela última crise**: os arquivos de features são ordenados
cronologicamente pelo campo `file_order` (salvo pelo NB3 v2). O conjunto de
treino contém todos os arquivos até o penúltimo arquivo com crise; o conjunto
de teste contém todos os arquivos a partir do último arquivo com crise.

Isso reproduz o cenário clínico real: o modelo aprende com o histórico do
paciente e é avaliado na sua capacidade de prever uma crise que ainda não
aconteceu durante o treinamento.

## Estrutura

| Seção | O que faz | Runs |
|---|---|---|
| **1** | Etapa 1 — janela × modelo em R5 (3 células independentes) | 504 |
| **1.5** | Análise — janela ótima, CIOPR, comparação estatística | 0 |
| **2** | Chance-level (Snyder et al. 2008) | 0 |
| **3** | Etapa 3 — degradação R3→R0, 3 modelos (3 células independentes) | 504 |
| **3.5** | Análise — curva de degradação, nível crítico | 0 |
| **4** | Interpretabilidade — SHAP + Permutation Importance | 0 |
| **Total** | | **1.008** |

Cada run é salva linha a linha em CSV incremental — retomada segura se
interrompido.


## 0. Dependências

In [ ]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'
%pip install -q numpy pandas matplotlib seaborn scikit-learn xgboost shap tqdm scipy statsmodels


## 1. Configuração

In [ ]:
import os, glob, gc, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, roc_curve
from sklearn.inspection import permutation_importance
import xgboost as xgb
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm
import IPython.display as ipd

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# ── Diretórios ──
ROOT_DIR    = 'data'
FEAT_DIR    = os.path.join(ROOT_DIR, 'features')
RESULTS_DIR = os.path.join(ROOT_DIR, 'results')
FIG_DIR     = os.path.join(RESULTS_DIR, 'figures', 'nb4')
LOG_DIR     = os.path.join(ROOT_DIR, 'logs')
for d in [RESULTS_DIR, FIG_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Pacientes (deve bater com NB2/NB3) ──
PATIENTS = {
    'CHBMIT'  : ['chb01','chb03','chb04','chb05','chb06','chb07',
                 'chb08','chb10','chb11','chb12','chb13','chb14'],
    'Siena'   : ['PN00','PN01','PN03','PN05','PN06','PN09',
                 'PN10','PN12','PN13','PN14','PN16','PN17'],
    'Mendeley': ['p10','p11','p12','p13','p14','p15'],
    'SeizeIT2': ['sub-001','sub-002','sub-004','sub-005',
                 'sub-007','sub-008','sub-011','sub-012',
                 'sub-035','sub-039','sub-047','sub-073'],
}
ALL_PATIENTS = [(ds, pat) for ds, pats in PATIENTS.items() for pat in pats]

LEVELS   = ['R5','R3','R2','R1','R0']
LEVEL_DS = {
    'R5': ['CHBMIT','Siena','Mendeley'],
    'R3': ['CHBMIT','Siena','Mendeley'],
    'R2': ['CHBMIT','Siena','Mendeley'],
    'R1': ['CHBMIT','Siena','Mendeley'],
    'R0': ['CHBMIT','Siena','Mendeley','SeizeIT2'],
}
WINDOWS_MIN  = [10, 15, 30, 45]
MODELS       = ['RF', 'XGB', 'LR']
N_FEAT_PER_CH = 19
STEP_SEC      = 15    # passo do NB3 v2 (segundos)
WIN_SEC       = 30
N_CONSEC      = 5     # janelas consecutivas para confirmar alarme

print('Configuração carregada.')
print(f'  {sum(len(v) for v in PATIENTS.values())} pacientes | '
      f'{len(LEVELS)} níveis | {WINDOWS_MIN} min | modelos: {MODELS}')


## 2. Funções compartilhadas

In [ ]:
# ── Manifesto do NB3 ──
_manifest_cache = None
def get_manifest():
    global _manifest_cache
    if _manifest_cache is None:
        mp = os.path.join(LOG_DIR, 'nb3_manifest.csv')
        if not os.path.exists(mp):
            raise FileNotFoundError(
                'nb3_manifest.csv não encontrado. Execute a célula de '
                'manifesto no NB3 v2 primeiro (célula "## 6. Manifesto global").')
        _manifest_cache = pd.read_csv(mp)
    return _manifest_cache


def load_patient_data(dataset, patient, level, window_min):
    '''Carrega todos os arquivos de features de um paciente, ordenados
    cronologicamente pelo field `file_order` salvo pelo NB3 v2.

    Retorna lista de dicts com:
      X_pre, X_inter, t_pre, t_inter, n_seizures, file_order, fkey
    '''
    df = get_manifest()
    mask = ((df['dataset'] == dataset) & (df['patient'] == patient) &
            (df['level'] == level) & (df['window_min'] == window_min))
    rows = df[mask].sort_values('file_order')
    if len(rows) == 0:
        return []

    files_data = []
    for _, row in rows.iterrows():
        fp = row['path']
        if not isinstance(fp, str) or not os.path.exists(fp):
            continue
        try:
            z = np.load(fp, allow_pickle=True)
            files_data.append(dict(
                fkey       = str(row['fkey']),
                file_order = int(row['file_order']),
                n_seizures = int(row['n_seizures']),
                n_pre      = int(row['n_pre']),
                n_inter    = int(row['n_inter']),
                X_pre      = z['X_pre'].copy()   if len(z['X_pre'])   > 0 else np.empty((0, z['X_pre'].shape[1]   if z['X_pre'].ndim > 1 else 0)),
                X_inter    = z['X_inter'].copy() if len(z['X_inter']) > 0 else np.empty((0, z['X_inter'].shape[1] if z['X_inter'].ndim > 1 else 0)),
                t_pre      = z['t_pre'].copy(),
                t_inter    = z['t_inter'].copy(),
            ))
            z.close()
        except Exception as e:
            print(f'  Aviso: erro ao carregar {fp}: {e}')
    return files_data


def split_by_last_seizure(files_data):
    '''Split pela última crise: treino = tudo antes da última crise,
    teste = último arquivo com crise + todos os posteriores.

    Retorna (train_files, test_files) ou (None, None) se inviável.
    '''
    if not files_data:
        return None, None

    # Encontra índice do último arquivo com crise
    last_seiz_idx = None
    for i, f in enumerate(files_data):
        if f['n_seizures'] > 0:
            last_seiz_idx = i

    if last_seiz_idx is None:
        return None, None   # nenhum arquivo tem crise (não deveria acontecer)

    if last_seiz_idx == 0:
        return None, None   # só 1 arquivo com crise e é o primeiro — sem treino

    train = files_data[:last_seiz_idx]   # tudo antes da última crise
    test  = files_data[last_seiz_idx:]   # última crise em diante

    # Verifica que o treino tem pelo menos 1 arquivo com crise (pré-ictal)
    if not any(f['n_seizures'] > 0 for f in train):
        return None, None   # treino sem pré-ictal

    # Verifica que o teste tem pré-ictal
    if not any(f['n_pre'] > 0 for f in test):
        return None, None   # teste sem pré-ictal

    return train, test


def build_Xy(files_list, mode='train', seed=42):
    '''Monta (X, y, dur_h) a partir de uma lista de arquivos.
    mode='train': balanceia 1:1 (N pré-ictais = N interictais amostrados)
                  e embaralha aleatoriamente.
    mode='test':  mantém distribuição natural, preserva ordem cronológica.
    '''
    all_pre   = [f['X_pre']   for f in files_list if len(f['X_pre'])   > 0]
    all_inter = [f['X_inter'] for f in files_list if len(f['X_inter']) > 0]
    all_t_inter = [f['t_inter'] for f in files_list if len(f['t_inter']) > 0]

    X_pre   = np.vstack(all_pre)   if all_pre   else np.empty((0,0))
    X_inter = np.vstack(all_inter) if all_inter else np.empty((0,0))
    t_inter = np.concatenate(all_t_inter) if all_t_inter else np.array([])

    # Duração total do teste em horas (para FP/h)
    total_janelas = len(X_pre) + len(X_inter)
    dur_h = total_janelas * STEP_SEC / 3600

    if len(X_pre) == 0 or len(X_inter) == 0:
        return None, None, dur_h

    if mode == 'train':
        # Amostragem estratificada temporal do interictal (1:1)
        n_target = len(X_pre)
        if len(X_inter) > n_target:
            rng = np.random.default_rng(seed)
            order = np.argsort(t_inter)
            bins  = np.array_split(order, n_target)
            pick  = np.array([rng.choice(b) for b in bins if len(b) > 0][:n_target])
            X_inter = X_inter[np.sort(pick)]
        X = np.vstack([X_inter, X_pre])
        y = np.concatenate([np.zeros(len(X_inter), dtype=np.int8),
                            np.ones(len(X_pre),    dtype=np.int8)])
        # Embaralha (necessário para modelos clássicos)
        idx = np.random.permutation(len(X))
        return X[idx], y[idx], dur_h
    else:
        # Teste: distribuição natural, ordem preservada
        X = np.vstack([X_inter, X_pre])
        y = np.concatenate([np.zeros(len(X_inter), dtype=np.int8),
                            np.ones(len(X_pre),    dtype=np.int8)])
        return X, y, dur_h


def get_model(name):
    if name == 'RF':
        return RandomForestClassifier(n_estimators=300, max_depth=15,
            min_samples_leaf=5, class_weight='balanced', n_jobs=2, random_state=42)
    if name == 'XGB':
        return xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, n_jobs=2, random_state=42,
            eval_metric='logloss', verbosity=0)
    if name == 'LR':
        return LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced',
            solver='lbfgs', n_jobs=1, random_state=42)
    raise ValueError(f'Modelo desconhecido: {name}')


def compute_metrics(y_true, y_prob, dur_h):
    if len(np.unique(y_true)) < 2 or len(y_true) == 0:
        return {}
    fpr_c, tpr_c, thrs = roc_curve(y_true, y_prob)
    thr = float(thrs[np.argmax(tpr_c - fpr_c)])
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if (tp+fn) > 0 else 0.0
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0.0
    prec = tp/(tp+fp) if (tp+fp) > 0 else 0.0
    f1   = 2*prec*sens/(prec+sens) if (prec+sens) > 0 else 0.0
    fph  = fp/dur_h if dur_h > 0 else 0.0
    try: auc = float(roc_auc_score(y_true, y_prob))
    except: auc = float('nan')
    # Sensibilidade com N consecutivas
    y_consec = (np.convolve(y_pred, np.ones(N_CONSEC), 'same') >= N_CONSEC).astype(int)
    tp_c = int(((y_consec==1)&(y_true==1)).sum())
    fp_c = int(((y_consec==1)&(y_true==0)).sum())
    sens_c = tp_c/(tp+fn) if (tp+fn) > 0 else 0.0
    fph_c  = fp_c/dur_h   if dur_h > 0  else 0.0
    return dict(auc=round(auc,4), f1=round(f1,4),
                sensitivity=round(sens,4), specificity=round(spec,4),
                fph=round(fph,4), threshold=round(thr,4),
                sensitivity_consec=round(sens_c,4), fph_consec=round(fph_c,4),
                tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))


def fit_eval(X_tr, y_tr, X_te, y_te, model_name, dur_h):
    scaler   = StandardScaler()
    X_tr_s   = scaler.fit_transform(X_tr)
    X_te_s   = scaler.transform(X_te)
    model    = get_model(model_name)
    t0       = time.time()
    model.fit(X_tr_s, y_tr)
    t_train  = time.time() - t0
    y_prob   = (model.predict_proba(X_te_s)[:,1]
                if hasattr(model,'predict_proba')
                else model.decision_function(X_te_s))
    mets = compute_metrics(y_te, y_prob, dur_h)
    return mets, round(t_train,2), model, scaler


def load_done(csv_path, key_cols):
    if not os.path.exists(csv_path): return set()
    try:
        df = pd.read_csv(csv_path)
        return set(zip(*[df[c].astype(str) for c in key_cols]))
    except: return set()


def save_row(row, csv_path):
    df = pd.DataFrame([row])
    df.to_csv(csv_path, mode='a', index=False, header=not os.path.exists(csv_path))


print('Funções compartilhadas definidas.')
print('Split: pela última crise (treino = histórico, teste = última crise em diante)')


## 3. Seção 1 — Etapa 1: janela × modelo × R5

Testa todas as combinações (janela pré-ictal × modelo) no nível R5 (17 canais,
informação completa) para todos os 42 pacientes.

**Uma célula por modelo** — independentes entre si. Cada uma pode ser rodada
em sessões diferentes. O CSV é salvo linha por linha (retomada segura).

Saída: `data/results/nb4_etapa1_{modelo}.csv`


### Seção 1 — RF

In [ ]:
# ── Seção 1 / RF ──
CSV1 = os.path.join(RESULTS_DIR, 'nb4_etapa1_RF.csv')
done = load_done(CSV1, ['dataset','patient','window_min'])

todo = [(ds, pat, w) for ds, pat in ALL_PATIENTS
        if ds in LEVEL_DS['R5']
        for w in WINDOWS_MIN
        if (ds, pat, str(w)) not in done]

print(f'Seção 1 / RF: {len(todo)} runs pendentes de '
      f'{len([p for p in ALL_PATIENTS if p[0] in LEVEL_DS["R5"]])*len(WINDOWS_MIN)} total')

pbar = tqdm(todo, desc='Etapa1/RF', ncols=100)
for ds, pat, w in pbar:
    pbar.set_postfix_str(f'{ds}|{pat}|w{w}')
    files_data = load_patient_data(ds, pat, 'R5', w)
    train_files, test_files = split_by_last_seizure(files_data)

    if train_files is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='RF',
                      level='R5', status='sem_split'), CSV1)
        continue

    X_tr, y_tr, _      = build_Xy(train_files, 'train')
    X_te, y_te, dur_h  = build_Xy(test_files,  'test')

    if X_tr is None or X_te is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='RF',
                      level='R5', status='sem_dados'), CSV1)
        continue

    try:
        mets, t_tr, _, _ = fit_eval(X_tr, y_tr, X_te, y_te, 'RF', dur_h)
        row = dict(dataset=ds, patient=pat, window_min=w, model='RF',
                   level='R5', status='ok', t_train_s=t_tr,
                   n_train=len(X_tr), n_test=len(X_te),
                   n_train_pre=int(y_tr.sum()), n_test_pre=int(y_te.sum()),
                   **mets)
    except Exception as e:
        row = dict(dataset=ds, patient=pat, window_min=w, model='RF',
                   level='R5', status=f'erro:{e}')
    save_row(row, CSV1)

# Resumo rápido
if os.path.exists(CSV1):
    df_tmp = pd.read_csv(CSV1)
    ok = df_tmp[df_tmp.get('status','ok')=='ok']
    if len(ok):
        print(f'\nRF Etapa 1 — AUC: {ok["auc"].mean():.3f}±{ok["auc"].std():.3f} | '
              f'F1: {ok["f1"].mean():.3f} | Sens: {ok["sensitivity"].mean():.3f} | '
              f'FP/h: {ok["fph"].mean():.2f}')


### Seção 1 — XGB

In [ ]:
# ── Seção 1 / XGB ──
CSV1 = os.path.join(RESULTS_DIR, 'nb4_etapa1_XGB.csv')
done = load_done(CSV1, ['dataset','patient','window_min'])

todo = [(ds, pat, w) for ds, pat in ALL_PATIENTS
        if ds in LEVEL_DS['R5']
        for w in WINDOWS_MIN
        if (ds, pat, str(w)) not in done]

print(f'Seção 1 / XGB: {len(todo)} runs pendentes de '
      f'{len([p for p in ALL_PATIENTS if p[0] in LEVEL_DS["R5"]])*len(WINDOWS_MIN)} total')

pbar = tqdm(todo, desc='Etapa1/XGB', ncols=100)
for ds, pat, w in pbar:
    pbar.set_postfix_str(f'{ds}|{pat}|w{w}')
    files_data = load_patient_data(ds, pat, 'R5', w)
    train_files, test_files = split_by_last_seizure(files_data)

    if train_files is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='XGB',
                      level='R5', status='sem_split'), CSV1)
        continue

    X_tr, y_tr, _      = build_Xy(train_files, 'train')
    X_te, y_te, dur_h  = build_Xy(test_files,  'test')

    if X_tr is None or X_te is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='XGB',
                      level='R5', status='sem_dados'), CSV1)
        continue

    try:
        mets, t_tr, _, _ = fit_eval(X_tr, y_tr, X_te, y_te, 'XGB', dur_h)
        row = dict(dataset=ds, patient=pat, window_min=w, model='XGB',
                   level='R5', status='ok', t_train_s=t_tr,
                   n_train=len(X_tr), n_test=len(X_te),
                   n_train_pre=int(y_tr.sum()), n_test_pre=int(y_te.sum()),
                   **mets)
    except Exception as e:
        row = dict(dataset=ds, patient=pat, window_min=w, model='XGB',
                   level='R5', status=f'erro:{e}')
    save_row(row, CSV1)

# Resumo rápido
if os.path.exists(CSV1):
    df_tmp = pd.read_csv(CSV1)
    ok = df_tmp[df_tmp.get('status','ok')=='ok']
    if len(ok):
        print(f'\nXGB Etapa 1 — AUC: {ok["auc"].mean():.3f}±{ok["auc"].std():.3f} | '
              f'F1: {ok["f1"].mean():.3f} | Sens: {ok["sensitivity"].mean():.3f} | '
              f'FP/h: {ok["fph"].mean():.2f}')


### Seção 1 — LR

In [ ]:
# ── Seção 1 / LR ──
CSV1 = os.path.join(RESULTS_DIR, 'nb4_etapa1_LR.csv')
done = load_done(CSV1, ['dataset','patient','window_min'])

todo = [(ds, pat, w) for ds, pat in ALL_PATIENTS
        if ds in LEVEL_DS['R5']
        for w in WINDOWS_MIN
        if (ds, pat, str(w)) not in done]

print(f'Seção 1 / LR: {len(todo)} runs pendentes de '
      f'{len([p for p in ALL_PATIENTS if p[0] in LEVEL_DS["R5"]])*len(WINDOWS_MIN)} total')

pbar = tqdm(todo, desc='Etapa1/LR', ncols=100)
for ds, pat, w in pbar:
    pbar.set_postfix_str(f'{ds}|{pat}|w{w}')
    files_data = load_patient_data(ds, pat, 'R5', w)
    train_files, test_files = split_by_last_seizure(files_data)

    if train_files is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='LR',
                      level='R5', status='sem_split'), CSV1)
        continue

    X_tr, y_tr, _      = build_Xy(train_files, 'train')
    X_te, y_te, dur_h  = build_Xy(test_files,  'test')

    if X_tr is None or X_te is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='LR',
                      level='R5', status='sem_dados'), CSV1)
        continue

    try:
        mets, t_tr, _, _ = fit_eval(X_tr, y_tr, X_te, y_te, 'LR', dur_h)
        row = dict(dataset=ds, patient=pat, window_min=w, model='LR',
                   level='R5', status='ok', t_train_s=t_tr,
                   n_train=len(X_tr), n_test=len(X_te),
                   n_train_pre=int(y_tr.sum()), n_test_pre=int(y_te.sum()),
                   **mets)
    except Exception as e:
        row = dict(dataset=ds, patient=pat, window_min=w, model='LR',
                   level='R5', status=f'erro:{e}')
    save_row(row, CSV1)

# Resumo rápido
if os.path.exists(CSV1):
    df_tmp = pd.read_csv(CSV1)
    ok = df_tmp[df_tmp.get('status','ok')=='ok']
    if len(ok):
        print(f'\nLR Etapa 1 — AUC: {ok["auc"].mean():.3f}±{ok["auc"].std():.3f} | '
              f'F1: {ok["f1"].mean():.3f} | Sens: {ok["sensitivity"].mean():.3f} | '
              f'FP/h: {ok["fph"].mean():.2f}')


## 4. Seção 1.5 — Análise da Etapa 1

Lê os CSVs da Seção 1. Não treina nenhum modelo.


In [ ]:
# Carrega todos os CSVs da Etapa 1
frames1 = []
for mdl in MODELS:
    fp = os.path.join(RESULTS_DIR, f'nb4_etapa1_{mdl}.csv')
    if os.path.exists(fp):
        tmp = pd.read_csv(fp); tmp['model'] = mdl; frames1.append(tmp)

if not frames1:
    print('Nenhum CSV da Etapa 1 encontrado. Rode a Seção 1 primeiro.')
else:
    df1 = pd.concat(frames1, ignore_index=True)
    df1_ok = df1[df1['status']=='ok'].copy()
    print(f'Etapa 1: {len(df1_ok)} runs válidas | '
          f'Datasets: {df1_ok["dataset"].unique().tolist()} | '
          f'Modelos: {df1_ok["model"].unique().tolist()}')

    # ── 4.1 Janela ótima por paciente ──
    best_w = (df1_ok.groupby(['dataset','patient','window_min'])['auc']
              .mean().reset_index()
              .sort_values('auc', ascending=False)
              .groupby(['dataset','patient']).first()
              .reset_index()[['dataset','patient','window_min']])

    print('\n--- Janela ótima por paciente (distribuição) ---')
    print(best_w.groupby(['dataset','window_min']).size().unstack(fill_value=0))

    fig, axes = plt.subplots(1, 2, figsize=(13,5))
    best_w['window_min'].value_counts().sort_index().plot(
        kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_xlabel('Janela pré-ictal (min)')
    axes[0].set_ylabel('N pacientes')
    axes[0].set_title('Distribuição da janela ótima por paciente')
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

    # AUC por modelo e janela
    pivot = df1_ok.groupby(['model','window_min'])['auc'].mean().unstack()
    pivot.plot(ax=axes[1], marker='o')
    axes[1].set_xlabel('Modelo')
    axes[1].set_ylabel('AUC médio')
    axes[1].set_title('AUC por modelo e janela')
    axes[1].legend(title='Janela (min)', bbox_to_anchor=(1.01,1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR,'etapa1_janela_modelo.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── 4.2 Comparação estatística entre modelos ──
    print('\n--- Wilcoxon pareado entre modelos (AUC, janela ótima por paciente) ---')
    auc_best = (df1_ok.merge(best_w, on=['dataset','patient','window_min'])
                .groupby(['dataset','patient','model'])['auc'].mean().reset_index())
    pairs = [('RF','XGB'),('RF','LR'),('XGB','LR')]
    p_vals, stat_rows = [], []
    for m1, m2 in pairs:
        a = auc_best[auc_best['model']==m1].set_index(['dataset','patient'])['auc']
        b = auc_best[auc_best['model']==m2].set_index(['dataset','patient'])['auc']
        common = a.index.intersection(b.index)
        if len(common) >= 5:
            stat, p = wilcoxon(a.loc[common], b.loc[common])
            p_vals.append(p)
            stat_rows.append((m1,m2,len(common),round(stat,2),round(p,4),None))
        else:
            stat_rows.append((m1,m2,len(common),None,None,None))
    if p_vals:
        _, p_corr, _, _ = multipletests(p_vals, method='holm')
        j = 0
        for i in range(len(stat_rows)):
            if stat_rows[i][4] is not None:
                stat_rows[i] = stat_rows[i][:5] + (round(p_corr[j],4),); j+=1
    df_stat = pd.DataFrame(stat_rows,
        columns=['modelo_A','modelo_B','n_pac','W','p_valor','p_corrigido'])
    ipd.display(df_stat)
    df_stat.to_csv(os.path.join(RESULTS_DIR,'nb4_comparacao_modelos.csv'), index=False)

    # ── 4.3 CIOPR proxy ──
    print('\n--- CIOPR proxy (sensibilidade × window_min = antecedência ponderada) ---')
    df1_ok['ciopr'] = df1_ok['sensitivity'] * df1_ok['window_min']
    ciopr = df1_ok.groupby(['model','window_min'])['ciopr'].mean().unstack().round(2)
    ipd.display(ciopr)
    ciopr.to_csv(os.path.join(RESULTS_DIR,'nb4_ciopr.csv'))

    # Salva janela ótima para uso nas seções seguintes
    best_w.to_csv(os.path.join(RESULTS_DIR,'nb4_best_window.csv'), index=False)
    print('\nCSVs salvos: nb4_comparacao_modelos.csv, nb4_ciopr.csv, nb4_best_window.csv')


## 5. Seção 2 — Chance-level (Snyder et al. 2008)

Preditor de Poisson calibrado para cada paciente. Requisito para publicação
em journal da área.


In [ ]:
def chance_level(fph_real, dur_h, n_pre_test, n_sim=2000, seed=42):
    '''Gera preditor aleatório com a mesma taxa de alarmes (FP/h) que o
    modelo real e calcula a sensibilidade média esperada por acaso.'''
    rng  = np.random.default_rng(seed)
    rate = min(fph_real * STEP_SEC / 3600, 1.0)   # prob. por janela de 30s
    n_janelas = int(dur_h * 3600 / STEP_SEC)
    if n_janelas == 0 or n_pre_test == 0:
        return float('nan')
    sens_sim = []
    for _ in range(n_sim):
        alarmes = (rng.random(n_janelas) < rate).astype(int)
        # assume pré-ictais uniformemente distribuídos no teste
        tp_r = min(alarmes.sum(), n_pre_test)
        sens_sim.append(tp_r / n_pre_test)
    return float(np.mean(sens_sim))


if 'df1_ok' not in dir() or len(df1_ok)==0:
    print('Rode a Seção 1.5 primeiro.')
else:
    if 'best_w' not in dir():
        best_w = pd.read_csv(os.path.join(RESULTS_DIR,'nb4_best_window.csv'))

    best_per_pat = (df1_ok.merge(best_w, on=['dataset','patient','window_min'])
                    .sort_values('auc', ascending=False)
                    .groupby(['dataset','patient']).first().reset_index())

    rows_chance = []
    for _, r in tqdm(best_per_pat.iterrows(), total=len(best_per_pat),
                     desc='Chance-level'):
        dur_h_est = ((r.get('n_test',0)) * STEP_SEC / 3600)
        sens_c = chance_level(r.get('fph',0.5), max(dur_h_est,0.1),
                              r.get('n_test_pre',1))
        rows_chance.append(dict(
            dataset=r['dataset'], patient=r['patient'],
            model=r['model'], window_min=r['window_min'],
            sens_real=r.get('sensitivity'), fph_real=r.get('fph'),
            auc_real=r.get('auc'), sens_chance=round(sens_c,4),
            supera_acaso=r.get('sensitivity',0) > sens_c
        ))

    df_chance = pd.DataFrame(rows_chance)
    pct = df_chance['supera_acaso'].mean()*100
    print(f'{pct:.1f}% dos pacientes: modelo real supera preditor aleatório')
    ipd.display(df_chance.groupby('dataset')['supera_acaso']
                .mean().rename('frac_supera_acaso').round(2))
    df_chance.to_csv(os.path.join(RESULTS_DIR,'nb4_chance_level.csv'), index=False)
    print('Salvo: nb4_chance_level.csv')


## 6. Seção 3 — Etapa 3: degradação por nível de canal

Usando a janela ótima de cada paciente (do CSV da Seção 1.5), testa os
3 modelos nos 4 níveis restantes (R3, R2, R1, R0). R5 já foi coberto na
Etapa 1.

**Uma célula por modelo** — independentes entre si.

Saída: `data/results/nb4_etapa3_{modelo}.csv`


### Seção 3 — RF

In [ ]:
# ── Seção 3 / RF ──
# Carrega janela ótima por paciente
if 'best_w' not in dir() or best_w is None:
    bp = os.path.join(RESULTS_DIR,'nb4_best_window.csv')
    best_w = pd.read_csv(bp) if os.path.exists(bp) else None
    if best_w is None:
        print('Rode a Seção 1.5 primeiro para gerar nb4_best_window.csv'); best_w = pd.DataFrame()

CSV3 = os.path.join(RESULTS_DIR, 'nb4_etapa3_RF.csv')
done3 = load_done(CSV3, ['dataset','patient','level'])
NIVEIS_REST = ['R3','R2','R1','R0']

todo3 = []
for _, brow in best_w.iterrows():
    ds, pat, w = brow['dataset'], brow['patient'], int(brow['window_min'])
    for lv in NIVEIS_REST:
        if ds not in LEVEL_DS.get(lv,[]):
            continue
        if (ds, pat, lv) not in done3:
            todo3.append((ds, pat, w, lv))

print(f'Seção 3 / RF: {len(todo3)} runs pendentes')

pbar = tqdm(todo3, desc='Etapa3/RF', ncols=100)
for ds, pat, w, lv in pbar:
    pbar.set_postfix_str(f'{lv}|{ds}|{pat}|w{w}')
    files_data  = load_patient_data(ds, pat, lv, w)
    train_files, test_files = split_by_last_seizure(files_data)

    if train_files is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='RF',
                      level=lv, status='sem_split'), CSV3)
        continue

    X_tr, y_tr, _     = build_Xy(train_files, 'train')
    X_te, y_te, dur_h = build_Xy(test_files,  'test')

    if X_tr is None or X_te is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='RF',
                      level=lv, status='sem_dados'), CSV3)
        continue

    try:
        mets, t_tr, _, _ = fit_eval(X_tr, y_tr, X_te, y_te, 'RF', dur_h)
        row = dict(dataset=ds, patient=pat, window_min=w, model='RF',
                   level=lv, status='ok', t_train_s=t_tr, **mets)
    except Exception as e:
        row = dict(dataset=ds, patient=pat, window_min=w, model='RF',
                   level=lv, status=f'erro:{e}')
    save_row(row, CSV3)

print(f'RF — Etapa 3 concluída.')


### Seção 3 — XGB

In [ ]:
# ── Seção 3 / XGB ──
# Carrega janela ótima por paciente
if 'best_w' not in dir() or best_w is None:
    bp = os.path.join(RESULTS_DIR,'nb4_best_window.csv')
    best_w = pd.read_csv(bp) if os.path.exists(bp) else None
    if best_w is None:
        print('Rode a Seção 1.5 primeiro para gerar nb4_best_window.csv'); best_w = pd.DataFrame()

CSV3 = os.path.join(RESULTS_DIR, 'nb4_etapa3_XGB.csv')
done3 = load_done(CSV3, ['dataset','patient','level'])
NIVEIS_REST = ['R3','R2','R1','R0']

todo3 = []
for _, brow in best_w.iterrows():
    ds, pat, w = brow['dataset'], brow['patient'], int(brow['window_min'])
    for lv in NIVEIS_REST:
        if ds not in LEVEL_DS.get(lv,[]):
            continue
        if (ds, pat, lv) not in done3:
            todo3.append((ds, pat, w, lv))

print(f'Seção 3 / XGB: {len(todo3)} runs pendentes')

pbar = tqdm(todo3, desc='Etapa3/XGB', ncols=100)
for ds, pat, w, lv in pbar:
    pbar.set_postfix_str(f'{lv}|{ds}|{pat}|w{w}')
    files_data  = load_patient_data(ds, pat, lv, w)
    train_files, test_files = split_by_last_seizure(files_data)

    if train_files is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='XGB',
                      level=lv, status='sem_split'), CSV3)
        continue

    X_tr, y_tr, _     = build_Xy(train_files, 'train')
    X_te, y_te, dur_h = build_Xy(test_files,  'test')

    if X_tr is None or X_te is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='XGB',
                      level=lv, status='sem_dados'), CSV3)
        continue

    try:
        mets, t_tr, _, _ = fit_eval(X_tr, y_tr, X_te, y_te, 'XGB', dur_h)
        row = dict(dataset=ds, patient=pat, window_min=w, model='XGB',
                   level=lv, status='ok', t_train_s=t_tr, **mets)
    except Exception as e:
        row = dict(dataset=ds, patient=pat, window_min=w, model='XGB',
                   level=lv, status=f'erro:{e}')
    save_row(row, CSV3)

print(f'XGB — Etapa 3 concluída.')


### Seção 3 — LR

In [ ]:
# ── Seção 3 / LR ──
# Carrega janela ótima por paciente
if 'best_w' not in dir() or best_w is None:
    bp = os.path.join(RESULTS_DIR,'nb4_best_window.csv')
    best_w = pd.read_csv(bp) if os.path.exists(bp) else None
    if best_w is None:
        print('Rode a Seção 1.5 primeiro para gerar nb4_best_window.csv'); best_w = pd.DataFrame()

CSV3 = os.path.join(RESULTS_DIR, 'nb4_etapa3_LR.csv')
done3 = load_done(CSV3, ['dataset','patient','level'])
NIVEIS_REST = ['R3','R2','R1','R0']

todo3 = []
for _, brow in best_w.iterrows():
    ds, pat, w = brow['dataset'], brow['patient'], int(brow['window_min'])
    for lv in NIVEIS_REST:
        if ds not in LEVEL_DS.get(lv,[]):
            continue
        if (ds, pat, lv) not in done3:
            todo3.append((ds, pat, w, lv))

print(f'Seção 3 / LR: {len(todo3)} runs pendentes')

pbar = tqdm(todo3, desc='Etapa3/LR', ncols=100)
for ds, pat, w, lv in pbar:
    pbar.set_postfix_str(f'{lv}|{ds}|{pat}|w{w}')
    files_data  = load_patient_data(ds, pat, lv, w)
    train_files, test_files = split_by_last_seizure(files_data)

    if train_files is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='LR',
                      level=lv, status='sem_split'), CSV3)
        continue

    X_tr, y_tr, _     = build_Xy(train_files, 'train')
    X_te, y_te, dur_h = build_Xy(test_files,  'test')

    if X_tr is None or X_te is None:
        save_row(dict(dataset=ds, patient=pat, window_min=w, model='LR',
                      level=lv, status='sem_dados'), CSV3)
        continue

    try:
        mets, t_tr, _, _ = fit_eval(X_tr, y_tr, X_te, y_te, 'LR', dur_h)
        row = dict(dataset=ds, patient=pat, window_min=w, model='LR',
                   level=lv, status='ok', t_train_s=t_tr, **mets)
    except Exception as e:
        row = dict(dataset=ds, patient=pat, window_min=w, model='LR',
                   level=lv, status=f'erro:{e}')
    save_row(row, CSV3)

print(f'LR — Etapa 3 concluída.')


## 7. Seção 3.5 — Análise de degradação de canal

In [ ]:
# Junta Etapa 1 (R5) e Etapa 3 (R3-R0)
frames_all = []
for mdl in MODELS:
    for etapa, lv_col in [('etapa1','R5'),('etapa3',None)]:
        fp = os.path.join(RESULTS_DIR, f'nb4_{etapa}_{mdl}.csv')
        if not os.path.exists(fp): continue
        tmp = pd.read_csv(fp)
        tmp = tmp[tmp.get('status','ok')=='ok'].copy()
        if lv_col: tmp['level'] = lv_col
        tmp['model'] = mdl
        frames_all.append(tmp)

if not frames_all:
    print('Rode as Seções 1 e 3 primeiro.')
else:
    df_all = pd.concat(frames_all, ignore_index=True)
    df_all = df_all[df_all.get('status','ok')=='ok'].copy()
    CH_MAP = {'R5':17,'R3':13,'R2':11,'R1':4,'R0':2}
    df_all['n_canais'] = df_all['level'].map(CH_MAP)

    fig, axes = plt.subplots(1,3, figsize=(15,5))
    nivel_order = ['R5','R3','R2','R1','R0']
    colors = {'RF':'steelblue','XGB':'darkorange','LR':'green'}

    for metric, ax, ylabel in [
        ('auc',         axes[0], 'AUC médio'),
        ('sensitivity', axes[1], 'Sensibilidade média'),
        ('fph',         axes[2], 'FP/h médio'),
    ]:
        for mdl in MODELS:
            sub = df_all[df_all['model']==mdl]
            grp = sub.groupby('level')[metric].mean()
            grp = grp.reindex([l for l in nivel_order if l in grp.index])
            x   = [CH_MAP[l] for l in grp.index]
            ax.plot(x, grp.values, marker='o', label=mdl, color=colors[mdl])
        ax.set_xlabel('Canais no nível'); ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel} por nível de canal')
        ax.set_xticks([17,13,11,4,2])
        ax.set_xticklabels(['R5\n(17)','R3\n(13)','R2\n(11)','R1\n(4)','R0\n(2)'])
        ax.legend(title='Modelo', bbox_to_anchor=(1.01,1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR,'etapa3_degradacao.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Wilcoxon entre níveis consecutivos
    print('\n--- Queda significativa entre níveis? (Wilcoxon, AUC) ---')
    rows_deg = []
    for mdl in MODELS:
        sub = df_all[df_all['model']==mdl]
        for i in range(len(nivel_order)-1):
            lvA, lvB = nivel_order[i], nivel_order[i+1]
            a = sub[sub['level']==lvA].set_index(['dataset','patient'])['auc']
            b = sub[sub['level']==lvB].set_index(['dataset','patient'])['auc']
            common = a.index.intersection(b.index)
            if len(common) >= 5:
                stat, p = wilcoxon(a.loc[common], b.loc[common])
                rows_deg.append(dict(model=mdl, nivel_A=lvA, nivel_B=lvB,
                                     n=len(common), W=round(stat,2),
                                     p=round(p,4), sig=(p<0.05)))
    df_deg = pd.DataFrame(rows_deg)
    ipd.display(df_deg)
    df_deg.to_csv(os.path.join(RESULTS_DIR,'nb4_degradacao_stat.csv'), index=False)
    print('Salvo: nb4_degradacao_stat.csv')


## 8. Seção 4 — Interpretabilidade (SHAP + Permutation Importance)

Para o modelo vencedor (maior AUC médio na janela ótima). Agrega por
região cerebral e por banda de frequência.


In [ ]:
import shap

# Identifica modelo e janela vencedores
if 'df1_ok' not in dir() or len(df1_ok)==0:
    frames_tmp = []
    for m in MODELS:
        fp = os.path.join(RESULTS_DIR,f'nb4_etapa1_{m}.csv')
        if os.path.exists(fp):
            tmp = pd.read_csv(fp); tmp['model']=m; frames_tmp.append(tmp)
    df1_ok = pd.concat(frames_tmp) if frames_tmp else pd.DataFrame()
    df1_ok = df1_ok[df1_ok.get('status','ok')=='ok'] if len(df1_ok) else df1_ok

if len(df1_ok)==0:
    print('Rode a Seção 1 primeiro.')
else:
    if 'best_w' not in dir():
        best_w = pd.read_csv(os.path.join(RESULTS_DIR,'nb4_best_window.csv'))

    best_overall = df1_ok.groupby(['model','window_min'])['auc'].mean().idxmax()
    best_model_name, best_window = best_overall
    print(f'Modelo vencedor: {best_model_name} | Janela ótima global: {best_window} min')

    # Monta dataset completo para SHAP
    Xs, ys, feat_labels = [], [], None
    for ds, pats in PATIENTS.items():
        if ds not in LEVEL_DS['R5']: continue
        for pat in pats:
            files_data = load_patient_data(ds, pat, 'R5', best_window)
            if not files_data: continue
            if feat_labels is None:
                # pega nomes de features do primeiro arquivo disponível
                df_m = get_manifest()
                row0 = df_m[(df_m['dataset']==ds)&(df_m['patient']==pat)&
                            (df_m['level']=='R5')&(df_m['window_min']==best_window)].iloc[0]
                z0 = np.load(row0['path'], allow_pickle=True)
                chs = list(z0.get('channels', z0.get('channel_labels', [])))
                fns = ['std','var','rms','line_len','mobility','skewness','kurtosis',
                       'delta','theta','alpha','beta','gamma','sp_entropy','beta_rel',
                       'dwt_d1','dwt_d2','dwt_d3','dwt_d4','complexity']
                feat_labels = [f'{ch}_{fn}' for ch in chs for fn in fns]
                z0.close()
            all_files = files_data
            X_all = np.vstack([f['X_pre']   for f in all_files if len(f['X_pre'])>0] +
                              [f['X_inter'] for f in all_files if len(f['X_inter'])>0])
            y_all = np.concatenate([
                np.ones (sum(len(f['X_pre'])   for f in all_files), dtype=np.int8),
                np.zeros(sum(len(f['X_inter']) for f in all_files), dtype=np.int8)])
            if len(X_all) > 0: Xs.append(X_all); ys.append(y_all)

    if Xs:
        X_full = np.vstack(Xs); y_full = np.concatenate(ys)
        scaler_s = StandardScaler()
        X_fs     = scaler_s.fit_transform(X_full)
        model_s  = get_model(best_model_name)
        model_s.fit(X_fs, y_full)

        # SHAP
        N_SHAP = min(800, len(X_fs))
        idx_s  = np.random.choice(len(X_fs), N_SHAP, replace=False)
        print(f'Calculando SHAP em {N_SHAP} amostras...')
        if best_model_name in ('RF','XGB'):
            expl   = shap.TreeExplainer(model_s)
            sv     = expl.shap_values(X_fs[idx_s])
            sv     = sv[1] if isinstance(sv,list) else sv
        else:
            expl   = shap.LinearExplainer(model_s, X_fs[idx_s])
            sv     = expl.shap_values(X_fs[idx_s])

        feat_imp = np.abs(sv).mean(axis=0)
        labels   = feat_labels if feat_labels and len(feat_labels)==len(feat_imp) else [f'f{i}' for i in range(len(feat_imp))]
        df_shap  = pd.DataFrame({'feature':labels,'shap':feat_imp}).sort_values('shap',ascending=False)
        df_shap.to_csv(os.path.join(RESULTS_DIR,'nb4_shap.csv'), index=False)

        # Top 20
        fig, ax = plt.subplots(figsize=(9,7))
        top20 = df_shap.head(20)
        ax.barh(top20['feature'][::-1], top20['shap'][::-1], color='steelblue')
        ax.set_xlabel('Importância SHAP média')
        ax.set_title(f'Top 20 features — {best_model_name}, janela {best_window} min')
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR,'shap_top20.png'), dpi=150, bbox_inches='tight')
        plt.show()

        # Por região
        REGIOES = ['frontal','temporal','central','parietal','occipital']
        df_shap['regiao'] = df_shap['feature'].apply(
            lambda x: next((r for r in REGIOES if r in x.lower()),'outro'))
        reg = df_shap.groupby('regiao')['shap'].mean().sort_values(ascending=False)
        print('\nImportância por região:'); print(reg.round(5))
        reg.to_csv(os.path.join(RESULTS_DIR,'nb4_shap_regiao.csv'))

        # Por banda
        BANDAS = ['delta','theta','alpha','beta','gamma']
        df_shap['banda'] = df_shap['feature'].apply(
            lambda x: next((b for b in BANDAS if b in x.lower()),'outro'))
        band = df_shap.groupby('banda')['shap'].mean().sort_values(ascending=False)
        print('\nImportância por banda:'); print(band.round(5))
        band.to_csv(os.path.join(RESULTS_DIR,'nb4_shap_banda.csv'))

        # Permutation Importance
        print('\nPermutation Importance (confirma SHAP)...')
        perm = permutation_importance(model_s, X_fs, y_full, n_repeats=10,
                                      random_state=42, n_jobs=2)
        df_perm = pd.DataFrame({'feature':labels,
                                 'perm_imp':perm.importances_mean,
                                 'perm_std':perm.importances_std})
        df_perm.sort_values('perm_imp',ascending=False,inplace=True)
        df_perm.to_csv(os.path.join(RESULTS_DIR,'nb4_perm_imp.csv'), index=False)
        print('Arquivos de interpretabilidade salvos.')


## 9. Resumo — arquivos gerados

In [ ]:
print('=== Arquivos gerados pelo NB4 ===')
csvs = [
    ('nb4_etapa1_RF.csv',       'Etapa 1 — resultados RF por paciente/janela'),
    ('nb4_etapa1_XGB.csv',      'Etapa 1 — resultados XGB'),
    ('nb4_etapa1_LR.csv',       'Etapa 1 — resultados LR'),
    ('nb4_best_window.csv',     'Janela ótima por paciente (usada na Etapa 3)'),
    ('nb4_comparacao_modelos.csv','Wilcoxon + Holm-Bonferroni entre modelos'),
    ('nb4_ciopr.csv',           'CIOPR proxy por modelo e janela'),
    ('nb4_chance_level.csv',    'Comparação contra preditor aleatório (Snyder 2008)'),
    ('nb4_etapa3_RF.csv',       'Etapa 3 — degradação de canal, RF'),
    ('nb4_etapa3_XGB.csv',      'Etapa 3 — degradação de canal, XGB'),
    ('nb4_etapa3_LR.csv',       'Etapa 3 — degradação de canal, LR'),
    ('nb4_degradacao_stat.csv', 'Wilcoxon entre níveis consecutivos'),
    ('nb4_shap.csv',            'SHAP por feature'),
    ('nb4_shap_regiao.csv',     'SHAP agregado por região cerebral'),
    ('nb4_shap_banda.csv',      'SHAP agregado por banda de frequência'),
    ('nb4_perm_imp.csv',        'Permutation Importance'),
]
for fname, desc in csvs:
    fp = os.path.join(RESULTS_DIR, fname)
    status = '✓' if os.path.exists(fp) else '✗'
    print(f'  {status} {fname:<40s} {desc}')
figs = glob.glob(os.path.join(FIG_DIR,'*.png'))
print(f'\nFiguras: {len(figs)}')
for f in sorted(figs): print(f'  {os.path.basename(f)}')
